In [56]:
import kagglehub
import pandas as pd
import os
from pprint import pprint

In [114]:
# Only download if the dataset doesn't already exist
if not os.path.exists("data/lgg-mri-segmentation"):
    print("Downloading dataset...")
    path = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")
else:
    print("Dataset already exists, skipping download.")
    path = "data/lgg-mri-segmentation"

In [115]:
path = path + "\kaggle_3m"

In [116]:
# Lists everything directly inside the folder
with os.scandir(path) as entries:
    patients = []
    for entry in entries:
        if entry.is_dir():
            patients.append(entry)
        else:
            name, ext = os.path.splitext(entry.name)
            if ext.lower() == ".csv":
                pt_data = pd.read_csv(entry.path)

In [117]:
print("There are", len(patients), "patients in this data set.")
print(pt_data.describe())

There are 110 patients in this data set.
       RNASeqCluster  MethylationCluster  miRNACluster   CNCluster  \
count      92.000000          109.000000    110.000000  108.000000   
mean        2.445652            3.678899      1.900000    1.722222   
std         1.180092            1.169684      0.789263    0.862872   
min         1.000000            1.000000      1.000000    1.000000   
25%         1.000000            3.000000      1.000000    1.000000   
50%         2.000000            4.000000      2.000000    1.000000   
75%         4.000000            5.000000      2.000000    3.000000   
max         4.000000            5.000000      4.000000    3.000000   

       RPPACluster  OncosignCluster  COCCluster  histological_type  \
count    98.000000       105.000000  110.000000         109.000000   
mean      2.367347         1.895238    1.763636           2.128440   
std       1.125045         0.663960    0.855927           0.850935   
min       1.000000         1.000000    1.000000 

In [123]:
print(pt_data.loc[0,])

Patient                      TCGA_CS_4941
RNASeqCluster                         2.0
MethylationCluster                    4.0
miRNACluster                            2
CNCluster                             2.0
RPPACluster                           NaN
OncosignCluster                       3.0
COCCluster                              2
histological_type                     1.0
neoplasm_histologic_grade             2.0
tumor_tissue_site                     1.0
laterality                            3.0
tumor_location                        2.0
gender                                2.0
age_at_initial_pathologic            67.0
race                                  3.0
ethnicity                             2.0
death01                               1.0
Name: 0, dtype: object


In [124]:
patient_imgs = {}

for patient in patients:
    if patient.name not in patient_imgs:
        patient_imgs[patient.name] = []
    
    # to temporarily hold files by base name
    temp_scans = {}
    
    with os.scandir(patient.path) as patient_files:
        for file in patient_files:
            name, ext = os.path.splitext(file.name)
            
            if ext.lower() != ".tif":
                 continue
                
            if "mask" in name:
                base_name = name.replace("_mask", "")
                # initialize inner dict if needed
                if base_name not in temp_scans:
                    temp_scans[base_name] = {}
                temp_scans[base_name]["mask"] = file.path
            else:
                if name not in temp_scans:
                    temp_scans[name] = {}
                temp_scans[name]["image"] = file.path
                
    for base_name, pair in temp_scans.items():
        if "image" in pair and "mask" in pair:
            patient_imgs[patient.name].append(pair)
        else:
            print(f"Incomplete pair: {patient.name} - {base_name}")

In [129]:
total_scans = 0
n = True

for patient, scans in patient_imgs.items():
    total_scans += len(scans)
    if n:
        print(f"\n{patient} — {len(scans)} scans")
        for i, scan in enumerate(scans):
            print(f"  Scan {i+1}:")
            print(f"    Image: {os.path.basename(scan['image'])}")
            print(f"    Mask:  {os.path.basename(scan['mask'])}")
    n = False


TCGA_CS_4941_19960909 — 23 scans
  Scan 1:
    Image: TCGA_CS_4941_19960909_1.tif
    Mask:  TCGA_CS_4941_19960909_1_mask.tif
  Scan 2:
    Image: TCGA_CS_4941_19960909_10.tif
    Mask:  TCGA_CS_4941_19960909_10_mask.tif
  Scan 3:
    Image: TCGA_CS_4941_19960909_11.tif
    Mask:  TCGA_CS_4941_19960909_11_mask.tif
  Scan 4:
    Image: TCGA_CS_4941_19960909_12.tif
    Mask:  TCGA_CS_4941_19960909_12_mask.tif
  Scan 5:
    Image: TCGA_CS_4941_19960909_13.tif
    Mask:  TCGA_CS_4941_19960909_13_mask.tif
  Scan 6:
    Image: TCGA_CS_4941_19960909_14.tif
    Mask:  TCGA_CS_4941_19960909_14_mask.tif
  Scan 7:
    Image: TCGA_CS_4941_19960909_15.tif
    Mask:  TCGA_CS_4941_19960909_15_mask.tif
  Scan 8:
    Image: TCGA_CS_4941_19960909_16.tif
    Mask:  TCGA_CS_4941_19960909_16_mask.tif
  Scan 9:
    Image: TCGA_CS_4941_19960909_17.tif
    Mask:  TCGA_CS_4941_19960909_17_mask.tif
  Scan 10:
    Image: TCGA_CS_4941_19960909_18.tif
    Mask:  TCGA_CS_4941_19960909_18_mask.tif
  Scan 11:
    Im

In [130]:
# Quick final sanity check before closing Notebook 1
print(f"Total patients:      {len(patient_imgs)}")
print(f"Total scan pairs:    {total_scans}")
print(f"Avg scans/patient:   {total_scans/len(patient_imgs):.1f}")
print(f"\nMetadata CSV shape:  {pt_data.shape}")
print(f"Metadata columns:    {list(pt_data.columns)}")

# Check for any patients with zero pairs
empty = [p for p, scans in patient_imgs.items() if len(scans) == 0]
if empty:
    print(f"\n⚠️ Patients with no scans: {empty}")
else:
    print(f"\n✅ All patients have scan pairs")

Total patients:      110
Total scan pairs:    3929
Avg scans/patient:   35.7

Metadata CSV shape:  (110, 18)
Metadata columns:    ['Patient', 'RNASeqCluster', 'MethylationCluster', 'miRNACluster', 'CNCluster', 'RPPACluster', 'OncosignCluster', 'COCCluster', 'histological_type', 'neoplasm_histologic_grade', 'tumor_tissue_site', 'laterality', 'tumor_location', 'gender', 'age_at_initial_pathologic', 'race', 'ethnicity', 'death01']

✅ All patients have scan pairs
